# Section A: DQ → MatMul Fusion Rules — Supporting Data

**Context:** ORT's `DQMatMulToMatMulNBits` graph rewriter only fuses `DequantizeLinear → MatMul` into `MatMulNBits` for **4-bit block-quantized FP32** models. This leaves two gaps:

1. **FP32 8-bit models** — No fusion rules exist for 8-bit, so DQ+MatMul remain separate ops.
2. **FP16 models (all bit widths)** — FP16 models introduce `Cast` ops around `MatMul` (since core `MatMul` doesn't support FP16), which block the DQ→MatMul pattern match entirely.

Below we quantify the performance cost by comparing **MNB** (native `MatMulNBits` op, the ideal fused target) vs **QDQ** (`DequantizeLinear → MatMul`, unfused) latencies, averaged across devices.

**Filters applied:**
- MNB: `format_type=mnb, symmetry=asym`
- QDQ: `format_type=qdq, symmetry=asym, signedness=unsigned, granularity=block, weight_layout=original, scenario=qdq_fused`
- For 8-bit, `qdq_fused` is effectively unfused since no 8-bit fusion rules exist. For FP16, Cast ops block all fusion.

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.1f}")

BASE_DIR = Path(r"C:\Users\jambaykinley\OneDrive - Microsoft\Desktop\work-ARM\llm-data")
MODEL_SIZE_ORDER = ["0.5b", "1.5b", "3b", "7b"]
SEQ_ORDER = [1, 128, 256, 512, 1024]


def load_fp32_data():
    """Load FP32 data from all_devices_summary.csv."""
    df = pd.read_csv(BASE_DIR / "qdq-cpu" / "all_devices_summary.csv")
    df["bits"] = df["bits"].astype(int)
    df["seq_length"] = df["seq_length"].astype(int)
    df["mean_ms"] = df["mean_ms"].astype(float)
    df = df[df["device"].isin(["amd", "intel"])]
    return df


def load_fp16_data():
    """Load FP16 data from per-device summary.csv files."""
    devices = {
        "amd": BASE_DIR / "qdq-cpu-fp16" / "hm_amd_laptop_perf",
        "intel": BASE_DIR / "qdq-cpu-fp16" / "intel_surface_laptop_perf",
    }
    frames = []
    for device, path in devices.items():
        tmp = pd.read_csv(path / "summary.csv")
        tmp["device"] = device
        frames.append(tmp)
    df = pd.concat(frames, ignore_index=True)
    df["bits"] = df["bits"].astype(int)
    df["seq_length"] = df["seq_length"].astype(int)
    df["mean_ms"] = df["mean_ms"].astype(float)
    return df


def compare_mnb_vs_qdq(df, bits):
    """Compare MNB vs QDQ, returning a compact table pivoted by seq_length.

    Rows: model_size
    Columns: mnb_ms and qdq/mnb ratio at each seq_length
    """
    mnb = df[
        (df["format_type"] == "mnb")
        & (df["bits"] == bits)
        & (df["symmetry"] == "asym")
    ]
    qdq = df[
        (df["format_type"] == "qdq")
        & (df["bits"] == bits)
        & (df["symmetry"] == "asym")
        & (df["granularity"] == "block")
        & (df["signedness"] == "unsigned")
        & (df["weight_layout"] == "original")
        & (df["scenario"] == "qdq_fused")
    ]

    mnb_avg = mnb.groupby(["model_size", "seq_length"])["mean_ms"].mean()
    qdq_avg = qdq.groupby(["model_size", "seq_length"])["mean_ms"].mean()
    ratio = qdq_avg / mnb_avg

    # Pivot to wide: rows=model_size, columns=seq_length
    mnb_wide = mnb_avg.unstack("seq_length").reindex(MODEL_SIZE_ORDER)[SEQ_ORDER]
    ratio_wide = ratio.unstack("seq_length").reindex(MODEL_SIZE_ORDER)[SEQ_ORDER]

    # Build compact multi-level columns
    mnb_wide.columns = pd.MultiIndex.from_tuples(
        [(f"seq={s}", "mnb (ms)") for s in SEQ_ORDER]
    )
    ratio_wide.columns = pd.MultiIndex.from_tuples(
        [(f"seq={s}", "qdq/mnb") for s in SEQ_ORDER]
    )

    # Interleave: for each seq_length, show mnb_ms then ratio
    result = pd.DataFrame(index=mnb_wide.index)
    result.index.name = "model"
    for s in SEQ_ORDER:
        col = f"seq={s}"
        result[(col, "mnb (ms)")] = mnb_wide[(col, "mnb (ms)")]
        result[(col, "qdq/mnb")] = ratio_wide[(col, "qdq/mnb")]
    result.columns = pd.MultiIndex.from_tuples(result.columns)
    return result


print("Helpers loaded.")

Helpers loaded.


## FP32: 8-bit Models — No Fusion Rules

The `DQMatMulToMatMulNBits` rewriter only handles 4-bit block quantization. For 8-bit models, `DequantizeLinear → MatMul` remains unfused even though a `MatMulNBits` kernel exists for 8-bit weights.

The table below compares latency of **MNB** (native 8-bit `MatMulNBits`) vs **QDQ** (unfused 8-bit `DQ → MatMul`), averaged across 2 devices (AMD, Intel).

In [10]:
df_fp32 = load_fp32_data()

print("FP32 8-bit: MNB (native) vs QDQ (unfused DQ→MatMul)")
print("Averaged across AMD + Intel\n")

tbl_fp32_8 = compare_mnb_vs_qdq(df_fp32, bits=8)
print(tbl_fp32_8.to_string(float_format=lambda x: f"{x:.1f}"))

FP32 8-bit: MNB (native) vs QDQ (unfused DQ→MatMul)
Averaged across AMD + Intel

         seq=1          seq=128          seq=256          seq=512         seq=1024        
      mnb (ms) qdq/mnb mnb (ms) qdq/mnb mnb (ms) qdq/mnb mnb (ms) qdq/mnb mnb (ms) qdq/mnb
model                                                                                     
0.5b      12.4    14.9    228.9     1.9    478.9     1.5    979.7     1.3   2113.6     1.3
1.5b      38.9    16.3    797.5     1.7   1542.8     1.4   3206.4     1.2   6987.1     1.1
3b        79.0    17.0   1558.2     1.8   3072.0     1.5   6147.3     1.3  13271.2     1.2
7b       166.0    17.8   3522.0     1.7   6886.2     1.4  13734.8     1.3  28092.1     1.2


## FP16 Models — Cast Ops Block All Fusion

FP16 models introduce `Cast` ops around `MatMul` (FP16→FP32 before, FP32→FP16 after) because the core `MatMul` operator does not support FP16. This breaks the `DQ → MatMul` pattern match, so **no fusion occurs — not even for 4-bit block quantization** which fuses successfully in FP32.

### FP16: 4-bit Models

This is especially notable: 4-bit block QDQ models fuse perfectly in FP32 (achieving MNB-equivalent performance), but in FP16 they remain completely unfused.

In [11]:
df_fp16 = load_fp16_data()

print("FP16 4-bit: MNB (native) vs QDQ (unfused — Cast ops block fusion)")
print("Averaged across AMD + Intel\n")

tbl_fp16_4 = compare_mnb_vs_qdq(df_fp16, bits=4)
print(tbl_fp16_4.to_string(float_format=lambda x: f"{x:.1f}"))

FP16 4-bit: MNB (native) vs QDQ (unfused — Cast ops block fusion)
Averaged across AMD + Intel

         seq=1          seq=128          seq=256          seq=512         seq=1024        
      mnb (ms) qdq/mnb mnb (ms) qdq/mnb mnb (ms) qdq/mnb mnb (ms) qdq/mnb mnb (ms) qdq/mnb
model                                                                                     
0.5b      10.0   275.2    293.5    10.2    609.0     5.3   1387.1     2.7   3042.7     1.6
1.5b      24.7   343.0    865.9    10.6   1833.4     5.4   3593.5     3.2   7546.1     2.0
3b        49.3   368.5   1691.8    11.5   3507.5     5.9   7034.6     3.4  14817.3     2.1
7b        97.6   475.0   3795.1    13.4   6850.0     8.3  13764.8     5.5  28653.6     4.1


### FP16: 8-bit Models

8-bit FP16 models are doubly impacted: no 8-bit fusion rules exist AND Cast ops would block fusion even if rules were added.

In [17]:
print("FP16 8-bit: MNB (native) vs QDQ (unfused — no 8-bit rules + Cast ops)")
print("Averaged across AMD + Intel\n")

tbl_fp16_8 = compare_mnb_vs_qdq(df_fp16, bits=8)
print(tbl_fp16_8.to_string(float_format=lambda x: f"{x:.1f}"))

FP16 8-bit: MNB (native) vs QDQ (unfused — no 8-bit rules + Cast ops)
Averaged across AMD + Intel

         seq=1          seq=128          seq=256          seq=512         seq=1024        
      mnb (ms) qdq/mnb mnb (ms) qdq/mnb mnb (ms) qdq/mnb mnb (ms) qdq/mnb mnb (ms) qdq/mnb
model                                                                                     
0.5b      14.5   125.3    317.4     6.5    638.3     3.5   1379.0     2.0   2934.6     1.4
1.5b      39.3   137.8    892.9     6.7   1878.4     3.5   3470.1     2.4   7430.5     1.6
3b        75.7   154.1   1557.8     8.4   3255.6     4.4   6409.0     2.7  13642.2     1.8


## Section B: Arm 8-bit MatMulNBits Kernel (FP16 scales)

Compare MNB 8-bit vs 4-bit latency ratio on ARM for FP32 and FP16 models. If the kernel is well-optimized, the 8-bit/4-bit ratio should be similar across precision. A higher ratio for FP16 8-bit indicates a missing or slower kernel path.

In [18]:
# Load ARM-only MNB data for FP32 and FP16
df_fp32_arm = pd.read_csv(BASE_DIR / "qdq-cpu" / "surface_arm_laptop_perf" / "summary.csv")
df_fp32_arm["bits"] = df_fp32_arm["bits"].astype(int)
df_fp32_arm["seq_length"] = df_fp32_arm["seq_length"].astype(int)
df_fp32_arm["mean_ms"] = df_fp32_arm["mean_ms"].astype(float)

df_fp16_arm = pd.read_csv(BASE_DIR / "qdq-cpu-fp16" / "surface_arm_laptop_perf" / "summary.csv")
df_fp16_arm["bits"] = df_fp16_arm["bits"].astype(int)
df_fp16_arm["seq_length"] = df_fp16_arm["seq_length"].astype(int)
df_fp16_arm["mean_ms"] = df_fp16_arm["mean_ms"].astype(float)


def arm_mnb_8vs4(df, label):
    """For ARM MNB asym data, compute 8-bit / 4-bit latency ratio."""
    mnb = df[(df["format_type"] == "mnb") & (df["symmetry"] == "asym")]
    mnb_4 = mnb[mnb["bits"] == 4].set_index(["model_size", "seq_length"])["mean_ms"]
    mnb_8 = mnb[mnb["bits"] == 8].set_index(["model_size", "seq_length"])["mean_ms"]

    # Pivot wide
    ms_4 = mnb_4.unstack("seq_length").reindex(MODEL_SIZE_ORDER)[SEQ_ORDER]
    ms_8 = mnb_8.unstack("seq_length").reindex(MODEL_SIZE_ORDER)[SEQ_ORDER]
    ratio = ms_8 / ms_4

    # Build interleaved table: 4-bit ms, 8-bit ms, 8b/4b ratio per seq
    result = pd.DataFrame(index=ms_4.index)
    result.index.name = "model"
    for s in SEQ_ORDER:
        result[(f"seq={s}", "4b (ms)")] = ms_4[s]
        result[(f"seq={s}", "8b (ms)")] = ms_8[s]
        result[(f"seq={s}", "8b/4b")] = ratio[s]
    result.columns = pd.MultiIndex.from_tuples(result.columns)
    return result


print(f"{'='*100}")
print(f"  ARM MNB: 8-bit vs 4-bit latency ratio — FP32 models")
print(f"{'='*100}\n")
tbl_arm_fp32 = arm_mnb_8vs4(df_fp32_arm, "FP32")
print(tbl_arm_fp32.to_string(float_format=lambda x: f"{x:.1f}"))

print(f"\n\n{'='*100}")
print(f"  ARM MNB: 8-bit vs 4-bit latency ratio — FP16 models")
print(f"{'='*100}\n")
tbl_arm_fp16 = arm_mnb_8vs4(df_fp16_arm, "FP16")
print(tbl_arm_fp16.to_string(float_format=lambda x: f"{x:.1f}"))

  ARM MNB: 8-bit vs 4-bit latency ratio — FP32 models

        seq=1               seq=128               seq=256               seq=512               seq=1024              
      4b (ms) 8b (ms) 8b/4b 4b (ms) 8b (ms) 8b/4b 4b (ms) 8b (ms) 8b/4b 4b (ms) 8b (ms) 8b/4b  4b (ms) 8b (ms) 8b/4b
model                                                                                                               
0.5b      9.7    46.5   4.8   317.2   290.0   0.9   763.9   717.4   0.9  1652.4  1376.4   0.8   3390.1  2348.6   0.7
1.5b     87.9    52.2   0.6  1331.4   645.8   0.5  2353.8  1228.9   0.5  4750.0  2258.9   0.5   8388.8  4970.4   0.6
3b      116.5   203.2   1.7  1923.1  1214.5   0.6  3922.6  2296.2   0.6  7311.0  4788.0   0.7  14182.5  9844.5   0.7


  ARM MNB: 8-bit vs 4-bit latency ratio — FP16 models

        seq=1               seq=128               seq=256               seq=512               seq=1024              
      4b (ms) 8b (ms) 8b/4b 4b (ms) 8b (ms) 8b/4b 4b (ms) 8b (ms) 8b/

## Section C: KleidiAI — Asymmetric vs Symmetric 4-bit on ARM

Compare MNB 4-bit asymmetric vs symmetric latency on ARM. KleidiAI's optimized packed kernel is disabled for asymmetric models, causing a fallback to generic NEON. A ratio > 1 means asymmetric is slower.

In [20]:
def arm_asym_vs_sym(df):
    """For ARM MNB 4-bit data, compute asym/sym latency ratio."""
    mnb_4 = df[(df["format_type"] == "mnb") & (df["bits"] == 4)]
    sym = mnb_4[mnb_4["symmetry"] == "sym"].set_index(["model_size", "seq_length"])["mean_ms"]
    asym = mnb_4[mnb_4["symmetry"] == "asym"].set_index(["model_size", "seq_length"])["mean_ms"]

    sizes = ["0.5b", "1.5b", "3b"]  # exclude 7b
    ms_sym = sym.unstack("seq_length").reindex(sizes)[SEQ_ORDER]
    ms_asym = asym.unstack("seq_length").reindex(sizes)[SEQ_ORDER]
    ratio = ms_asym / ms_sym

    # Build compact table: sym ms then asym/sym per seq
    result = pd.DataFrame(index=ms_sym.index)
    result.index.name = "model"
    for s in SEQ_ORDER:
        result[(f"seq={s}", "sym (ms)")] = ms_sym[s]
        result[(f"seq={s}", "asym/sym")] = ratio[s]
    result.columns = pd.MultiIndex.from_tuples(result.columns)
    return result


print("ARM MNB 4-bit: Asymmetric vs Symmetric — FP32\n")
tbl_c_fp32 = arm_asym_vs_sym(df_fp32_arm)
print(tbl_c_fp32.to_string(float_format=lambda x: f"{x:.2f}"))

print(f"\n\nARM MNB 4-bit: Asymmetric vs Symmetric — FP16\n")
tbl_c_fp16 = arm_asym_vs_sym(df_fp16_arm)
print(tbl_c_fp16.to_string(float_format=lambda x: f"{x:.2f}"))

ARM MNB 4-bit: Asymmetric vs Symmetric — FP32

         seq=1           seq=128           seq=256           seq=512          seq=1024         
      sym (ms) asym/sym sym (ms) asym/sym sym (ms) asym/sym sym (ms) asym/sym sym (ms) asym/sym
model                                                                                          
0.5b     29.08     0.33   539.23     0.59  1013.13     0.75  1943.19     0.85  3034.98     1.12
1.5b     64.27     1.37   841.36     1.58  1565.42     1.50  3063.15     1.55  6081.28     1.38
3b       66.90     1.74  1442.18     1.33  2612.12     1.50  5078.30     1.44 10086.84     1.41


ARM MNB 4-bit: Asymmetric vs Symmetric — FP16

         seq=1           seq=128           seq=256           seq=512          seq=1024         
      sym (ms) asym/sym sym (ms) asym/sym sym (ms) asym/sym sym (ms) asym/sym sym (ms) asym/sym
model                                                                                          
0.5b     20.29     0.52   600.79     0.6

## Section 3: FP16 Models — FP16 vs FP32 Comparison (AMD + Intel avg)

In [32]:
def compare_fp16_vs_fp32(df_fp32, df_fp16, bits, format_type):
    """Compare FP16 vs FP32 latency for a given format_type and bits."""
    if format_type == "mnb":
        filt = lambda d: d[
            (d["format_type"] == "mnb")
            & (d["bits"] == bits)
            & (d["symmetry"] == "asym")
        ]
    else:
        # 4-bit has qdq_unfused; 8-bit only has qdq_fused (which is effectively unfused)
        scenario = "qdq_unfused" if bits == 4 else "qdq_fused"
        filt = lambda d: d[
            (d["format_type"] == "qdq")
            & (d["bits"] == bits)
            & (d["symmetry"] == "asym")
            & (d["granularity"] == "block")
            & (d["signedness"] == "unsigned")
            & (d["weight_layout"] == "original")
            & (d["scenario"] == scenario)
        ]

    fp32 = filt(df_fp32).groupby(["model_size", "seq_length"])["mean_ms"].mean()
    fp16 = filt(df_fp16).groupby(["model_size", "seq_length"])["mean_ms"].mean()
    ratio = fp16 / fp32

    fp32_wide = fp32.unstack("seq_length").reindex(MODEL_SIZE_ORDER)[SEQ_ORDER]
    ratio_wide = ratio.unstack("seq_length").reindex(MODEL_SIZE_ORDER)[SEQ_ORDER]

    fp32_wide.columns = pd.MultiIndex.from_tuples(
        [(f"seq={s}", "fp32 (ms)") for s in SEQ_ORDER]
    )
    ratio_wide.columns = pd.MultiIndex.from_tuples(
        [(f"seq={s}", "fp16/fp32") for s in SEQ_ORDER]
    )

    result = pd.DataFrame(index=fp32_wide.index)
    result.index.name = "model"
    for s in SEQ_ORDER:
        col = f"seq={s}"
        result[(col, "fp32 (ms)")] = fp32_wide[(col, "fp32 (ms)")]
        result[(col, "fp16/fp32")] = ratio_wide[(col, "fp16/fp32")]
    result.columns = pd.MultiIndex.from_tuples(result.columns)
    return result

print("Helper loaded.")

Helper loaded.


In [23]:
print("=== MNB Asym 4-bit: FP16 / FP32 ===")
tbl_mnb_4 = compare_fp16_vs_fp32(df_fp32, df_fp16, 4, "mnb")
print(tbl_mnb_4.to_string())

=== MNB Asym 4-bit: FP16 / FP32 ===
          seq=1             seq=128             seq=256             seq=512            seq=1024          
      fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32
model                                                                                                    
0.5b        8.0       1.2     265.9       1.1     466.1       1.3     912.1       1.5    2184.5       1.4
1.5b       21.0       1.2     690.4       1.3    1354.0       1.4    2784.6       1.3    5963.5       1.3
3b         44.2       1.1    1641.6       1.0    3207.7       1.1    6477.9       1.1   13474.5       1.1


In [24]:
print("=== MNB Asym 8-bit: FP16 / FP32 ===")
tbl_mnb_8 = compare_fp16_vs_fp32(df_fp32, df_fp16, 8, "mnb")
print(tbl_mnb_8.to_string())

=== MNB Asym 8-bit: FP16 / FP32 ===
          seq=1             seq=128             seq=256             seq=512            seq=1024          
      fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32
model                                                                                                    
0.5b       12.4       1.2     228.9       1.4     478.9       1.3     979.7       1.4    2113.6       1.4
1.5b       38.9       1.0     797.5       1.1    1542.8       1.2    3206.4       1.1    6987.1       1.1
3b         79.0       1.0    1558.2       1.0    3072.0       1.1    6147.3       1.0   13271.2       1.0


In [33]:
print("=== QDQ Unfused Asym Unsigned 4-bit: FP16 / FP32 ===")
tbl_qdq_4 = compare_fp16_vs_fp32(df_fp32, df_fp16, 4, "qdq")
print(tbl_qdq_4.to_string())

=== QDQ Unfused Asym Unsigned 4-bit: FP16 / FP32 ===
          seq=1             seq=128             seq=256             seq=512            seq=1024          
      fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32
model                                                                                                    
0.5b      686.0       4.1     944.6       3.2    1213.6       2.6    1782.5       2.1    3172.6       1.7
1.5b     2322.9       3.7    3176.8       2.9    3890.2       2.6    5772.6       2.0    9488.0       1.6
3b       4566.8       3.9    6060.4       3.3    7718.3       2.8   11490.4       2.1   19918.6       1.6


In [34]:
print("=== QDQ Unfused Asym Unsigned 8-bit: FP16 / FP32 ===")
tbl_qdq_8 = compare_fp16_vs_fp32(df_fp32, df_fp16, 8, "qdq")
print(tbl_qdq_8.to_string())

=== QDQ Unfused Asym Unsigned 8-bit: FP16 / FP32 ===
          seq=1             seq=128             seq=256             seq=512            seq=1024          
      fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32 fp32 (ms) fp16/fp32
model                                                                                                    
0.5b      185.0       9.8     445.4       4.6     715.7       3.2    1298.1       2.2    2732.3       1.5
1.5b      634.1       8.5    1337.4       4.5    2104.1       3.2    3835.0       2.2    7649.0       1.5
3b       1340.9       8.7    2847.0       4.6    4598.2       3.1    8263.6       2.1   16510.9       1.5


In [31]:
# Check what combinations exist for QDQ unfused
for label, df in [("FP32", df_fp32), ("FP16", df_fp16)]:
    qdq = df[(df["format_type"] == "qdq") & (df["scenario"] == "qdq_unfused")]
    print(f"{label} qdq_unfused bits:", sorted(qdq["bits"].unique()))

FP32 qdq_unfused bits: [np.int64(4)]
FP16 qdq_unfused bits: [np.int64(4)]
